**Experiment 1 (revised)** -- synthetic proxy-discrimination demonstration with 30-seed 95% confidence intervals. Run top to bottom.

In [1]:
# 01_synthetic_proxy_discrimination.ipynb
#
# Experiment 1 -- Synthetic demonstration of proxy discrimination and its mitigation
#
# Purpose:
#   Demonstrate the central claim on fully synthetic data (no real personal data):
#   removing a protected attribute alone does NOT remove disparate impact, while
#   the governance fairness engine (group-wise threshold calibration) restores
#   parity and preserves utility. This revised version reports results over 30
#   random seeds with 95% confidence intervals (not single point estimates).
#
# Conditions: with_protected, drop_protected_only, drop_proxies_too, gov_engine
# Metrics: AUC, DIR, equalized-odds gaps (dTPR, dFPR)
# Outputs:
#   data/synthetic_credit.csv
#   results/tables/exp1_metrics.csv           (mean +/- 95% CI over seeds)
#   results/figures/exp1_auc_dir.(png|pdf)    (empirical scatter, seed 42)
#   results/figures/exp1_dir_by_condition.(png|pdf)  (bar with CI over seeds)

# ==== Imports and grayscale academic style (600 dpi, PNG+PDF, no captions) ====
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
FIG_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
for d in (DATA_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85"})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

def disparate_impact_ratio(y_pred, group):
    approve = (y_pred == 0).astype(int)
    r1 = approve[group == 1].mean(); r0 = approve[group == 0].mean()
    return min(r1, r0) / max(r1, r0) if max(r1, r0) > 0 else np.nan

def equalized_odds_gaps(y_true, y_pred, group):
    def rate(ct, mask):
        idx = (y_true == ct) & mask
        return (y_pred[idx] == 1).mean() if idx.sum() else np.nan
    a, b = group == 1, group == 0
    return abs(rate(1, a) - rate(1, b)), abs(rate(0, a) - rate(0, b))



# ==== Synthetic data generator (parameterized by seed) ====
def generate(seed):
    rng = np.random.default_rng(seed)
    N = 8000
    protected = rng.binomial(1, 0.35, N)
    residential_region = 1.6 * protected + rng.normal(0, 0.6, N)
    spending_pattern = 1.4 * protected + rng.normal(0, 0.6, N)
    income = rng.normal(0, 1, N); debt_ratio = rng.normal(0, 1, N); pay_history = rng.normal(0, 1, N)
    true_risk = -0.9*income + 0.8*debt_ratio - 0.7*pay_history
    bias_term = 1.3*residential_region + 1.1*spending_pattern
    logit = true_risk + bias_term + rng.normal(0, 0.5, N)
    default = (logit > np.quantile(logit, 0.7)).astype(int)
    return pd.DataFrame({"protected": protected,
        "residential_region": residential_region, "spending_pattern": spending_pattern,
        "income": income, "debt_ratio": debt_ratio, "pay_history": pay_history,
        "default": default})

FEATURE_SETS = {
    "with_protected": ["protected", "residential_region", "spending_pattern", "income", "debt_ratio", "pay_history"],
    "drop_protected_only": ["residential_region", "spending_pattern", "income", "debt_ratio", "pay_history"],
    "drop_proxies_too": ["income", "debt_ratio", "pay_history"],
}

def evaluate(df, feats, seed, gov=False):
    N = len(df)
    idx_tr, idx_te = train_test_split(np.arange(N), test_size=0.3, random_state=seed)
    g = df["protected"].values[idx_te]; y = df["default"].values[idx_te]
    Xtr = df[feats].values[idx_tr]; Xte = df[feats].values[idx_te]
    model = LogisticRegression(max_iter=1000).fit(Xtr, df["default"].values[idx_tr])
    proba = model.predict_proba(Xte)[:, 1]; auc = roc_auc_score(y, proba)
    if not gov:
        yp = (proba >= 0.5).astype(int)
    else:
        base = ((proba >= 0.5).astype(int) == 0).mean()
        t1 = np.quantile(proba[g == 1], base); t0 = np.quantile(proba[g == 0], base)
        yp = np.where(g == 1, (proba >= t1), (proba >= t0)).astype(int)
    dtpr, dfpr = equalized_odds_gaps(y, yp, g)
    return auc, disparate_impact_ratio(yp, g), dtpr, dfpr

# ==== Multi-seed evaluation (30 seeds, 95% CI) ====
SEEDS = list(range(30))
spec = [("with_protected", FEATURE_SETS["with_protected"], False),
        ("drop_protected_only", FEATURE_SETS["drop_protected_only"], False),
        ("drop_proxies_too", FEATURE_SETS["drop_proxies_too"], False),
        ("gov_engine", FEATURE_SETS["drop_protected_only"], True)]

rows = []
for name, feats, gov in spec:
    arr = np.array([evaluate(generate(s), feats, s, gov) for s in SEEDS])
    m = arr.mean(0); ci = 1.96 * arr.std(0) / np.sqrt(len(SEEDS))
    rows.append({"condition": name, "AUC": m[0], "AUC_ci": ci[0],
                 "DIR": m[1], "DIR_ci": ci[1], "dTPR": m[2], "dTPR_ci": ci[2],
                 "dFPR": m[3], "dFPR_ci": ci[3]})
results = pd.DataFrame(rows)
results.to_csv(os.path.join(TAB_DIR, "exp1_metrics.csv"), index=False)
print(results.round(3).to_string(index=False))

# Persist one representative synthetic dataset (seed 42) for reproducibility
generate(42).to_csv(os.path.join(DATA_DIR, "synthetic_credit.csv"), index=False)

# ==== Figure 1: AUC vs DIR scatter (representative seed 42) ====
df42 = generate(42)
pts = {name: evaluate(df42, feats, 42, gov)[:2] for name, feats, gov in spec}
order = ["with_protected", "drop_protected_only", "drop_proxies_too", "gov_engine"]
markers = ["o", "s", "^", "D"]; shades = ["#000000", "#555555", "#999999", "#cccccc"]
fig, ax = plt.subplots(figsize=(6, 4.2))
for i, name in enumerate(order):
    auc, dirv = pts[name]
    ax.scatter(dirv, auc, s=110, marker=markers[i], facecolor=shades[i],
               edgecolor="black", linewidth=1.0, label=name, zorder=3)
ax.axvline(0.8, color="black", linestyle="--", linewidth=0.9, zorder=1)
ax.set_xlabel("Disparate Impact Ratio (DIR)"); ax.set_ylabel("AUC")
ax.set_xlim(0.0, 1.05); ax.legend(frameon=True, edgecolor="black", loc="lower left")
fig.tight_layout(); save_fig(fig, "exp1_auc_dir"); plt.close(fig)

# ==== Figure 2: DIR by condition with 95% CI over seeds ====
pl = results.set_index("condition").loc[order].reset_index()
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.bar(range(len(pl)), pl["DIR"], yerr=pl["DIR_ci"], capsize=4,
       edgecolor="black", linewidth=1.0, color=shades)
ax.axhline(0.8, color="black", linestyle="--", linewidth=0.9)
ax.set_xticks(range(len(pl))); ax.set_xticklabels(pl["condition"], rotation=20, ha="right")
ax.set_ylabel("Disparate Impact Ratio (DIR)"); ax.set_ylim(0, 1.05)
fig.tight_layout(); save_fig(fig, "exp1_dir_by_condition"); plt.close(fig)
print("Saved tables and figures.")


          condition   AUC  AUC_ci   DIR  DIR_ci  dTPR  dTPR_ci  dFPR  dFPR_ci
     with_protected 0.992   0.000 0.298   0.006 0.214    0.017 0.135    0.009
drop_protected_only 0.992   0.000 0.299   0.006 0.210    0.017 0.133    0.008
   drop_proxies_too 0.718   0.004 0.986   0.004 0.511    0.019 0.089    0.004
         gov_engine 0.992   0.000 1.000   0.000 0.586    0.003 0.239    0.003


Saved tables and figures.
